In [1]:
import random
import math

# Cities as (x, y) coordinates
cities = {
    0: (0, 0),
    1: (2, 4),
    2: (5, 2),
    3: (8, 6),
    4: (6, 0),
    5: (3, 7)
}

NUM_CITIES    = len(cities)
POP_SIZE      = 100
GENERATIONS   = 500
MUTATION_RATE = 0.01

# ── Distance between two cities ──────────────────────
def distance(c1, c2):
    x1, y1 = cities[c1]
    x2, y2 = cities[c2]
    return math.sqrt((x2-x1)**2 + (y2-y1)**2)

# ── Total route distance ─────────────────────────────
def route_distance(route):
    total = 0
    for i in range(len(route)):
        total += distance(route[i], route[(i+1) % NUM_CITIES])
    return total

# ── Fitness = 1/distance (shorter = fitter) ──────────
def fitness(route):
    return 1 / route_distance(route)

# ── Create initial random population ─────────────────
def initial_population():
    population = []
    for _ in range(POP_SIZE):
        route = list(range(NUM_CITIES))
        random.shuffle(route)
        population.append(route)
    return population

# ── Select parent using tournament selection ─────────
def select(population):
    tournament = random.sample(population, 5)  # pick 5 random
    return max(tournament, key=fitness)         # best among 5 wins

# ── Ordered crossover (OX) ────────────────────────────
def crossover(parent1, parent2):
    start = random.randint(0, NUM_CITIES - 2)
    end   = random.randint(start + 1, NUM_CITIES)

    child = [-1] * NUM_CITIES
    child[start:end] = parent1[start:end]  # copy slice from parent1

    # fill remaining from parent2 in order
    pos = end
    for city in parent2:
        if city not in child:
            if pos >= NUM_CITIES:
                pos = 0
            child[pos] = city
            pos += 1
    return child

# ── Mutation: swap two random cities ─────────────────
def mutate(route):
    if random.random() < MUTATION_RATE:
        i, j = random.sample(range(NUM_CITIES), 2)
        route[i], route[j] = route[j], route[i]
    return route

# ── Main GA loop ──────────────────────────────────────
population = initial_population()
best_route = min(population, key=route_distance)

for gen in range(GENERATIONS):
    new_population = []

    for _ in range(POP_SIZE):
        parent1 = select(population)
        parent2 = select(population)
        child   = crossover(parent1, parent2)
        child   = mutate(child)
        new_population.append(child)

    population = new_population

    # track best route
    current_best = min(population, key=route_distance)
    if route_distance(current_best) < route_distance(best_route):
        best_route = current_best

    if gen % 100 == 0:
        print(f"Gen {gen} | Best Distance: {route_distance(best_route):.2f}")

print(f"\nBest Route   : {best_route}")
print(f"Total Distance: {route_distance(best_route):.2f}")

Gen 0 | Best Distance: 25.97
Gen 100 | Best Distance: 25.97
Gen 200 | Best Distance: 25.97
Gen 300 | Best Distance: 25.97
Gen 400 | Best Distance: 25.97

Best Route   : [3, 2, 4, 0, 1, 5]
Total Distance: 25.97
